# 05 - Training set + fraud model (step 6)

* **Training set:** point-in-time join of the 13 registered Feature Views (`04_feature_views`) onto every silver transfer, plus one **request-time** input the app sends with each scoring call: `payout_method` (`RequestSource`). `amount_usd` is deliberately left out: its single-feature AUC on this synthetic data is 0.994, because the generator sets fraud amounts to 4–15× normal, so including it would mostly measure the generator rather than the behavioral features. The 6 `anchor_*` features are freshness helpers and are **not** model inputs.
* **Label:** `silver_transfers.is_fraud` (synthetic ground truth).
* **Split:** temporal. Transfers have a real event time and scoring predicts future transfers, so the latest ~20% of time is held out. Features are always computed from full history up to each row's time.
* **Model:** a scikit-learn `Pipeline`. Preprocessing lives inside the model: numeric NULL becomes 0 (an empty window), decimals are cast to float, `payout_method` is one-hot encoded, and `last_10_screens` becomes count and risky-screen share (accepting the offline array or the online JSON string). The classifier is LightGBM with class weighting.
* **Metrics:** ROC-AUC, PR-AUC, precision and recall at the top 2% of scores (analyst review capacity), log-loss; all logged to MLflow.
* **Logging:** after holdout evaluation, the pipeline is refit on all transfers and logged with `fe.log_model`, so serving looks features up automatically from the Lakebase online store. Registration and the serving endpoint follow after confirmation.

In [0]:
%pip install -q "databricks-feature-engineering>=0.17" lightgbm uv
dbutils.library.restartPython()

In [0]:
import json, math
import numpy as np, pandas as pd
import mlflow
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient
from databricks.feature_engineering.entities import RequestSource, FieldDefinition, ScalarDataType, ColumnSelection, Feature

dbutils.widgets.text("catalog", "mlo_serverless_stable_catalog")
dbutils.widgets.text("schema", "fraud_rt")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
SEED = 42

USER = spark.sql("SELECT current_user()").first()[0]
EXPERIMENT = f"/Users/{USER}/rt-fraud-fs-model"
mlflow.set_experiment(EXPERIMENT)
mlflow.set_registry_uri("databricks-uc")
fe = FeatureEngineeringClient()
print(CATALOG, SCHEMA, EXPERIMENT)

In [0]:
FV_NAMES = ["failed_otp_count_10m", "failed_otp_count_1h", "failed_login_count_1h", "login_count_1h",
            "distinct_devices_24h", "screen_view_count_10m", "avg_dwell_ms_10m", "last_10_screens",
            "transfer_sum_7d", "transfer_sum_30d", "transfer_count_30d", "corridor_transfer_count_30d",
            "distinct_corridors_30d"]
feature_views = [fe.get_feature(full_name=f"{CATALOG}.{SCHEMA}.{n}") for n in FV_NAMES]

# Sent by the app with every scoring request (known at transfer time, not looked up).
# amount_usd is deliberately NOT a model input: in this synthetic data its single-feature AUC is 0.994 because the
# generator sets fraud amounts to 4-15x the customer's usual amount, so it would mostly measure the generator.
request_src = RequestSource(schema=[FieldDefinition(name="payout_method", data_type=ScalarDataType.STRING)])
def get_or_register(feature):
    # fe.log_model requires every feature (including request-time ones) to be registered in Unity Catalog.
    try:
        return fe.get_feature(full_name=f"{CATALOG}.{SCHEMA}.{feature.name}")
    except Exception:
        return fe.register_feature(feature=feature, catalog_name=CATALOG, schema_name=SCHEMA)

request_features = [get_or_register(Feature(source=request_src, function=ColumnSelection("payout_method"), name="payout_method"))]
MODEL_FEATURES = feature_views + request_features
INPUT_COLS = FV_NAMES + ["payout_method"]
KEY_COLS = ["transfer_id", "customer_id", "corridor", "ts", "event_ts"]
print(len(MODEL_FEATURES), "model inputs:", INPUT_COLS)

In [0]:
# One row per transfer. Behavioral features use event_ts and transfer features use ts, so the frame carries both.
labeled = (spark.table(f"{CATALOG}.{SCHEMA}.silver_transfers")
           .select("transfer_id", "customer_id", "corridor", "ts", F.col("ts").alias("event_ts"),
                   "payout_method", "is_fraud"))

# Analysis frame keeps keys and timestamps for the temporal split and diagnostics.
analysis_set = fe.create_training_set(df=labeled, features=MODEL_FEATURES, label="is_fraud")
df = analysis_set.load_df().toPandas()
df["ts"] = pd.to_datetime(df["ts"])
print(df.shape)
display(spark.createDataFrame(df.drop(columns=["last_10_screens"]).head(5).astype(str)))

In [0]:
counts = df["is_fraud"].value_counts().sort_index()
shares = (counts / counts.sum()).round(4)
print(pd.DataFrame({"transfers": counts, "share": shares}))

ftypes = []
for c in INPUT_COLS:
    kind = ("list of strings (screen names)" if c == "last_10_screens" else
            "categorical / nominal" if c == "payout_method" else
            "numeric float" if c in ("amount_usd", "avg_dwell_ms_10m", "transfer_sum_7d", "transfer_sum_30d") else
            "numeric integer (count)")
    ftypes.append((c, kind))
display(spark.createDataFrame(ftypes, "feature string, type string"))

NUM_COLS = [c for c in INPUT_COLS if c not in ("payout_method", "last_10_screens")]
num = df[NUM_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)
display(spark.createDataFrame(num.describe().T.round(2).reset_index().rename(columns={"index": "feature"})))

In [0]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

y_all = df["is_fraud"].astype(int)
corr = num.corrwith(y_all).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
corr.plot.barh(ax=axes[0], title="Correlation with is_fraud")
im = axes[1].imshow(num.corr().values, cmap="coolwarm", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(NUM_COLS)), NUM_COLS, rotation=90, fontsize=7)
axes[1].set_yticks(range(len(NUM_COLS)), NUM_COLS, fontsize=7)
axes[1].set_title("Feature correlogram")
fig.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()

# Fraud rate by payout method (request-time categorical input)
display(spark.createDataFrame(df.groupby("payout_method")["is_fraud"].agg(["count", "mean"]).round(4).reset_index()))

# Leakage screen: single-feature AUC. Anything near 1.0 would need a check that it is known at transfer time.
single_auc = {c: round(max(roc_auc_score(y_all, num[c]), 1 - roc_auc_score(y_all, num[c])), 3) for c in NUM_COLS}
display(spark.createDataFrame(sorted(single_auc.items(), key=lambda kv: -kv[1]), "feature string, single_feature_auc double"))

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from lightgbm import LGBMClassifier

RISKY_SCREENS = {"security_settings", "change_phone", "add_recipient", "limits"}

def to_float(X):
    """Feature tables return DECIMAL sums (Decimal objects); cast every numeric input to float."""
    return pd.DataFrame(X).apply(pd.to_numeric, errors="coerce").astype(float)

def screen_stats(X):
    """last_10_screens -> [n_screens, risky_share]. Offline it is an array, online a JSON string; NULL = no screens."""
    col = X.iloc[:, 0] if hasattr(X, "iloc") else np.asarray(X)[:, 0]
    out = []
    for v in col:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            lst = []
        elif isinstance(v, str):
            lst = json.loads(v)
        else:
            lst = list(v)
        n = len(lst)
        out.append([n, (sum(s in RISKY_SCREENS for s in lst) / n) if n else 0.0])
    return np.asarray(out, dtype=float)

NUMERIC = [c for c in INPUT_COLS if c not in ("payout_method", "last_10_screens")]

def build_pipeline(scale_pos_weight):
    pre = ColumnTransformer([
        ("num", Pipeline([("float", FunctionTransformer(to_float)),
                          ("zero_fill", SimpleImputer(strategy="constant", fill_value=0.0))]), NUMERIC),
        ("payout", OneHotEncoder(handle_unknown="ignore"), ["payout_method"]),
        ("screens", FunctionTransformer(screen_stats), ["last_10_screens"]),
    ], remainder="drop")
    clf = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                         subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                         scale_pos_weight=scale_pos_weight, random_state=SEED, verbose=-1)
    return Pipeline([("preprocess", pre), ("model", clf)])

from sklearn.base import BaseEstimator, ClassifierMixin

class FraudScorer(BaseEstimator, ClassifierMixin):
    """Wraps the pipeline so predict() returns P(fraud). The serving endpoint calls predict(), and the review queue
    needs a score it can rank and threshold, not a 0/1 label."""
    def __init__(self, pipeline=None):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        self.classes_ = self.pipeline.classes_
        return self
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)
    def predict(self, X):
        return self.pipeline.predict_proba(X)[:, 1]

# Zero-fill is deliberate: a NULL window aggregate means "no events in the window", i.e. 0, not "unknown".
print("numeric:", NUMERIC)

In [0]:
# ts is the real transfer time; holding out the latest period mirrors scoring future transfers.
cutoff = df["ts"].quantile(0.8)
train_df, test_df = df[df["ts"] < cutoff], df[df["ts"] >= cutoff]
X_train, y_train = train_df[INPUT_COLS], train_df["is_fraud"].astype(int)
X_test, y_test = test_df[INPUT_COLS], test_df["is_fraud"].astype(int)
spw = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f"cutoff={cutoff}  train={len(train_df)} ({y_train.sum()} fraud)  test={len(test_df)} ({y_test.sum()} fraud)  scale_pos_weight={spw:.1f}")

In [0]:
from sklearn.metrics import average_precision_score, log_loss, roc_curve, precision_recall_curve

def precision_recall_at_top(y_true, scores, frac=0.02):
    k = max(1, math.ceil(frac * len(scores)))
    top = np.argsort(-scores)[:k]
    hits = np.asarray(y_true)[top].sum()
    return hits / k, hits / max(np.asarray(y_true).sum(), 1), k

with mlflow.start_run(run_name="lgbm_feature_views_v1") as run:
    pipe = build_pipeline(spw)
    pipe.fit(X_train, y_train)
    p_test = pipe.predict_proba(X_test)[:, 1]
    p_at2, r_at2, k2 = precision_recall_at_top(y_test, p_test, 0.02)
    metrics = {"test_roc_auc": roc_auc_score(y_test, p_test), "test_pr_auc": average_precision_score(y_test, p_test),
               "test_precision_at_top2pct": p_at2, "test_recall_at_top2pct": r_at2,
               "test_log_loss": log_loss(y_test, p_test), "test_fraud_rate": float(y_test.mean())}
    mlflow.log_params({"model": "LGBMClassifier", "n_estimators": 400, "learning_rate": 0.05, "num_leaves": 31,
                       "scale_pos_weight": round(spw, 2), "split": "temporal_80_20", "cutoff": str(cutoff),
                       "seed": SEED, "n_train": len(X_train), "n_test": len(X_test), "top_k_at_2pct": k2,
                       "inputs": ",".join(INPUT_COLS)})
    mlflow.log_metrics(metrics)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    fpr, tpr, _ = roc_curve(y_test, p_test); ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "--")
    ax[0].set_title(f"ROC (AUC={metrics['test_roc_auc']:.3f})"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
    pr, rc, _ = precision_recall_curve(y_test, p_test); ax[1].plot(rc, pr)
    ax[1].set_title(f"PR (AP={metrics['test_pr_auc']:.3f})"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
    plt.tight_layout(); mlflow.log_figure(fig, "holdout_roc_pr.png"); plt.show()

    payout_levels = pipe.named_steps["preprocess"].named_transformers_["payout"].categories_[0]
    names = NUMERIC + [f"payout={c}" for c in payout_levels] + ["n_screens", "risky_screen_share"]
    imp = pd.Series(pipe.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False)
    mlflow.log_dict(imp.to_dict(), "feature_importance.json")
    RUN_ID = run.info.run_id

print(json.dumps({k: round(v, 4) for k, v in metrics.items()}, indent=1))
display(spark.createDataFrame(imp.reset_index().rename(columns={"index": "feature", 0: "importance"})))

In [0]:
from mlflow.models import infer_signature

# The model must be trained on TrainingSet.load_df() of the set passed to fe.log_model (no keys/timestamps).
final_set = fe.create_training_set(df=labeled, features=MODEL_FEATURES, label="is_fraud", exclude_columns=KEY_COLS)
final_pd = final_set.load_df().toPandas()
X_final, y_final = final_pd[INPUT_COLS], final_pd["is_fraud"].astype(int)
spw_final = float((y_final == 0).sum() / max((y_final == 1).sum(), 1))
FINAL_MODEL = FraudScorer(build_pipeline(spw_final)).fit(X_final, y_final)
final_pipe = FINAL_MODEL.pipeline

# Plain-Python input example (arrays -> lists) for the isolated serving-readiness check.
example = X_final.head(5).copy()
example["last_10_screens"] = example["last_10_screens"].apply(lambda v: list(v) if v is not None else None)
for c in NUMERIC:
    example[c] = pd.to_numeric(example[c], errors="coerce").astype(float)

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_params({"n_final_fit": len(X_final), "scale_pos_weight_final": round(spw_final, 2)})
    # Feature-store packaged model: serving looks up features by customer_id (+ corridor) and timestamps.
    fe.log_model(model=FINAL_MODEL, artifact_path="model", flavor=mlflow.sklearn, training_set=final_set,
                 infer_input_example=True)
    # Plain pipeline artifact with signature + example, used only for the isolated dependency check.
    raw_info = mlflow.sklearn.log_model(FINAL_MODEL, name="raw_pipeline",
                                        signature=infer_signature(example, FINAL_MODEL.predict(example)),
                                        input_example=example)
FE_MODEL_URI = f"runs:/{RUN_ID}/model"
print("feature-store model:", FE_MODEL_URI, "| raw pipeline:", raw_info.model_uri)

In [0]:
# Installs the logged pip requirements from scratch with uv and predicts on the stored input example.
raw_model = mlflow.pyfunc.load_model(raw_info.model_uri)
assert raw_model.metadata.signature is not None and raw_model.input_example is not None
mlflow.models.predict(model_uri=raw_info.model_uri, input_data=raw_model.input_example, env_manager="uv")
print("isolated predict OK")

# predict() must return P(fraud), equal to predict_proba[:, 1], with values in [0, 1].
p = np.asarray(raw_model.predict(example), dtype=float)
assert np.allclose(p, final_pipe.predict_proba(example)[:, 1]) and ((p >= 0) & (p <= 1)).all()
print("predict() returns probabilities:", np.round(p, 4))

# Same check on the online-store representation of last_10_screens (JSON string) to rule out train/serve skew.
online_like = example.copy()
online_like["last_10_screens"] = online_like["last_10_screens"].apply(lambda v: json.dumps(v) if v is not None else None)
assert np.allclose(FINAL_MODEL.predict(example), FINAL_MODEL.predict(online_like))
print("array vs JSON-string screens give identical scores")

In [0]:
REGISTERED_MODEL = f"{CATALOG}.{SCHEMA}.transfer_fraud_model"
registered_version = mlflow.register_model(model_uri=FE_MODEL_URI, name=REGISTERED_MODEL, await_registration_for=300)
MODEL_VERSION = registered_version.version
print(REGISTERED_MODEL, "version", MODEL_VERSION, f"models:/{REGISTERED_MODEL}/{MODEL_VERSION}")

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
workload_response = w.api_client.do("GET", "/api/2.0/serving-endpoints:workload-configs")
supported = {c["workload_type"]: [s["key"] for s in c.get("workload_sizes", [])] for c in workload_response.get("workload_configs", [])}
print(supported)
# Small LightGBM + sklearn pipeline: CPU, smallest size, scale to zero (first call after idle is a cold start).
WORKLOAD_TYPE = "CPU"
WORKLOAD_SIZE = "Small" if "Small" in supported.get("CPU", []) else supported["CPU"][0]
SCALE_TO_ZERO = True
ENDPOINT = "transfer-fraud-model"
print(ENDPOINT, WORKLOAD_TYPE, WORKLOAD_SIZE, SCALE_TO_ZERO)

In [0]:
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput, ServingModelWorkloadType, EndpointTag

served_entity = ServedEntityInput(entity_name=REGISTERED_MODEL, entity_version=str(MODEL_VERSION),
                                  workload_type=ServingModelWorkloadType(WORKLOAD_TYPE), workload_size=WORKLOAD_SIZE,
                                  scale_to_zero_enabled=SCALE_TO_ZERO)
try:
    existing = w.serving_endpoints.get(ENDPOINT)
except NotFound:
    existing = None
if existing is None:
    w.serving_endpoints.create(name=ENDPOINT, config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[served_entity]),
                               tags=[EndpointTag(key="project", value="fe_bar_fraud_rt")])
    print("creating", ENDPOINT)
else:
    names = [e.entity_name for e in (existing.config.served_entities or [])]
    assert names == [REGISTERED_MODEL], f"refusing to replace endpoint serving {names}"
    w.serving_endpoints.update_config(name=ENDPOINT, served_entities=[served_entity])
    print("updating", ENDPOINT)

In [0]:
import time
from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady

def wait_for_endpoint_ready(name, timeout_s=1800, poll_s=20):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        st = w.serving_endpoints.get(name).state
        if st.ready == EndpointStateReady.READY and st.config_update == EndpointStateConfigUpdate.NOT_UPDATING:
            return st
        if st.config_update in (EndpointStateConfigUpdate.UPDATE_FAILED, EndpointStateConfigUpdate.UPDATE_CANCELED):
            raise RuntimeError(f"{name} deployment failed: {st.config_update.value}")
        time.sleep(poll_s)
    raise TimeoutError(f"{name} not ready after {timeout_s}s")

print(wait_for_endpoint_ready(ENDPOINT))

In [0]:
# Recent live transfers (all fraud + a sample of legit) whose features are in the current online windows.
recent = spark.table(f"{CATALOG}.{SCHEMA}.silver_transfers").where("ts >= current_timestamp() - INTERVAL 2 HOURS")
sample = (recent.where("is_fraud = 1").unionByName(recent.where("is_fraud = 0").orderBy(F.rand(SEED)).limit(20))
          .select("transfer_id", "customer_id", "corridor", "ts", "payout_method", "is_fraud").toPandas())
print(len(sample), "transfers,", int(sample.is_fraud.sum()), "fraud")

# 1) Score through the endpoint: the app sends only keys, timestamps and request-time inputs.
records = [{"customer_id": r.customer_id, "corridor": r.corridor, "ts": r.ts.isoformat(), "event_ts": r.ts.isoformat(),
            "payout_method": r.payout_method} for r in sample.itertuples()]
resp = w.serving_endpoints.query(name=ENDPOINT, dataframe_records=records)
sample["endpoint_score"] = np.asarray(resp.predictions, dtype=float)

# 2) Independent check: read the same customers' rows straight from the online tables and apply the model locally.
online_tables = {}
for n in FV_NAMES:
    for mf in fe.list_materialized_features(feature_name=f"{CATALOG}.{SCHEMA}.{n}"):
        if mf.is_online:
            online_tables.setdefault(mf.as_dict()["table_name"], []).append(n)
X_online = sample[["transfer_id", "customer_id", "corridor", "payout_method"]].copy()
ids = list(sample.customer_id.unique())
for tbl, feats in online_tables.items():
    t = spark.table(tbl).where(F.col("customer_id").isin(ids))
    keys = ["customer_id", "corridor"] if "corridor" in t.columns else ["customer_id"]
    X_online = X_online.merge(t.select(*keys, *feats).toPandas(), on=keys, how="left")
sample["local_score_on_online_features"] = FINAL_MODEL.predict(X_online[INPUT_COLS])
sample["abs_diff"] = (sample.endpoint_score - sample.local_score_on_online_features).abs()
print("max |endpoint - local| =", sample.abs_diff.max())
print("mean score: fraud =", round(sample[sample.is_fraud == 1].endpoint_score.mean(), 4),
      "| legit =", round(sample[sample.is_fraud == 0].endpoint_score.mean(), 4))
display(spark.createDataFrame(sample.sort_values("endpoint_score", ascending=False).astype(str)))